# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:")
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Note: The Croissant schema defines data using `@id` fields. We'll enumerate record sets (tables/views), and for each, list available fields and their unique `@id`s.

In [ ]:
# List available record sets and their fields using @id

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")

for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    fields = list(rs.fields)
    print(f"  Fields ({len(fields)}):")
    for f in fields:
        print(f"    - {f.name} (@id: {f.id}, type: {f.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

We'll extract data from all available record sets into pandas DataFrames, indexed by record set `@id`.

In [ ]:
# Prepare DataFrames for each record set (@id referenced)

dataframes = {}
for rs in record_sets:
    print(f"Loading records from record set: {rs.name} (@id: {rs.id})...")
    records = list(dataset.records(record_set=rs.id))
    if len(records) == 0:
        print(f"  No records found for: {rs.id}\n")
        continue
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Loaded {len(df)} records. Columns:")
    print(f"    {list(df.columns)}\n")

if len(dataframes) > 0:
    # Show the first few rows of the first DataFrame loaded
    first_rs_id = list(dataframes.keys())[0]
    print(f"Head of DataFrame for {first_rs_id}:")
    display(dataframes[first_rs_id].head())
else:
    print('No record sets with data available.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

> **Note:** We'll choose a record set with numeric fields, filter by a threshold, normalize it, and (if possible) group by a categorical field. All entities are referenced by their `@id`.

In [ ]:
# Pick a record set for EDA (using @id), for example, the first available
if len(dataframes) > 0:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Exploring record set: {rs_id}")
    print("Columns available:", list(df.columns))
    
    # Attempt to auto-select a numeric field (type 'Float' or 'Integer')
    record_set_obj = [rs for rs in record_sets if rs.id == rs_id][0]
    numeric_fields = [f for f in record_set_obj.fields if f.data_type in ['Float','Integer','Number']]
    if len(numeric_fields) == 0:
        print("No numeric fields available for EDA in this record set.")
    else:
        field = numeric_fields[0]
        numeric_field = field.id
        print(f"Using numeric field: {field.name} (@id: {numeric_field})")
        if numeric_field not in df.columns:
            print(f"Field {numeric_field} not found in DataFrame columns.")
        else:
            threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
            display(filtered_df.head())

            # Normalize selected numeric field
            normalized_col = f"{numeric_field}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records (showing first 5):")
            display(filtered_df[[numeric_field, normalized_col]].head())

            # Attempt to group by a non-numeric field, if available
            group_by_candidates = [f for f in record_set_obj.fields if f.data_type not in ['Float','Integer','Number']]
            if len(group_by_candidates) > 0:
                group_field = group_by_candidates[0].id
                print(f"Grouping by: {group_field}")
                if group_field in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                    print(f"Mean {numeric_field} by {group_field}:")
                    display(grouped_df.head())
                else:
                    print(f"Group field {group_field} not present in DataFrame.")
            else:
                print("No suitable group-by fields found.")
else:
    print("No record set with data to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> We'll plot a histogram for the chosen numeric field, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(7,5))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('Histogram not shown: no suitable numeric field found or no data loaded.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset was successfully loaded and inspected using the `mlcroissant` library.
- Record sets and fields were discovered via their `@id` identifiers, allowing for dynamic and precise data referencing.
- Exploratory analyses can be performed programmatically by referencing data entities via `@id`, making the workflow reproducible and modular.
- Depending on the data structure, deeper statistical or modeling analyses can proceed using this foundation.